# Grimm Brothers: Preprocessing

This notebook converts the plain text Gutenberg file into a DOCS table suitable for tokenizaton.

## Set Up

In [1]:
import pandas as pd
import numpy as np

In [2]:
data_home = "../input"
output_dir = "../working"
OHCO = ['doc_id', 'para_num', 'sent_num', 'token_num']

## Source file to LINES

In [3]:
pg_id = 25291
pg_title ="grimm"

In [4]:
# From original notebook
# text_file = f"/kaggle/input/datasets/gegeorge250/grimm-brothers/pg2591.txt"
# csv_file  = f"{output_dir}/{pg_title}.csv" # The file we will create
# LINES = pd.DataFrame(open(text_file, 'r', encoding='utf-8-sig').readlines(), columns=['line_str'])
# LINES.index.name = 'line_num'
# LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()

I didn't have access to your data directory, so I downloaded a new copy.

In [5]:
source_file = f"{output_dir}/pg2591.txt" 

# Download the file if we don't have a local copy
import os
if not os.path.exists(source_file) or os.path.getsize(source_file) == 0:
    print(f"\U0001F914 File {source_file} not found.\nDownloading ...")
    import requests
    source_url = "https://www.gutenberg.org/cache/epub/2591/pg2591.txt"
    r = requests.get(source_url)
    with open(source_file, "w") as outfile:
        outfile.write(r.text)
    print("\U0001F44D Done.")
    print("File size =", os.path.getsize(source_file))
else:
    print(f"\U0001F600 File {source_file} of size {os.path.getsize(source_file)} exists.")

🤔 File ../working/pg2591.txt not found.
👍 Done.
File size = 560168


In [6]:
# !more {source_file}
# !rm {source_file}

In [7]:
LINES = pd.DataFrame({'line_str':open(source_file, "r").readlines()})
LINES.index.name = 'line_num'
LINES.line_str = LINES.line_str.str.replace(r'\n+', ' ', regex=True).str.strip()
LINES.to_csv("../working/pg2591-LINES.csv", index=True)
LINES

,line_str
line_num,
0,﻿The Project Gutenberg eBook of Grimms' Fairy ...
1,
2,This eBook is for the use of anyone anywhere i...
3,most other parts of the world at no cost and w...
4,"whatsoever. You may copy it, give it away or r..."
...,...
9614,including how to make donations to the Project...
9615,"Archive Foundation, how to help produce our ne..."
9616,subscribe to our email newsletter to hear abou...


Get the title.

In [8]:
LINES.loc[0].line_str

"\ufeffThe Project Gutenberg eBook of Grimms' Fairy Tales"

In [9]:
title = LINES.loc[0].line_str.replace('The Project Gutenberg eBook of ', '')
print(title)

﻿Grimms' Fairy Tales


Identify the story breaks.

In [10]:
clip_pats = [
    r"^THE BROTHERS GRIMM FAIRY TALES",
    r'^\*\*\*\*\*'
]
pat_a = LINES.line_str.str.match(clip_pats[0])
pat_b = LINES.line_str.str.match(clip_pats[1])
line_a = LINES.loc[pat_a].index[0] + 1
line_b = LINES.loc[pat_b].index[0] - 1
line_a,line_b                                 

(np.int64(121), np.int64(9241))

In [11]:
LINES = LINES.loc[line_a : line_b].copy()
LINES.head(10)

,line_str
line_num,
121,
122,
123,
124,
125,THE GOLDEN BIRD
126,
127,
128,"A certain king had a beautiful garden, and in ..."
129,which bore golden apples. These apples were al...


## LINES to DOCS

Define pattern for story titles.

In [12]:
story_pat = r"^[A-Z][A-Z\s,'\-]{5,}$"
story_lines = LINES.line_str.str.match(story_pat, na=False) # Used AI to help match titles
# story_lines

Validate pattern by eye.

In [13]:
LINES.loc[story_lines] # Use as filter for dataframe

,line_str
line_num,
125,THE GOLDEN BIRD
334,HANS IN LUCK
539,JORINDA AND JORINDEL
658,THE TRAVELLING MUSICIANS
773,OLD SULTAN
...,...
8157,THE STORY OF THE YOUTH WHO WENT FORTH TO LEARN...
8463,KING GRISLY-BEARD
8593,IRON HANS


Apply pattern to identify rows with titles.

In [14]:
LINES.loc[story_lines, 'doc_id'] = [i+1 for i in range(LINES.loc[story_lines].shape[0])]
LINES

,line_str,doc_id
line_num,,
121,,NaN
122,,NaN
123,,NaN
124,,NaN
125,THE GOLDEN BIRD,1.0
...,...,...
9237,her children for many years. She took the two ...,NaN
9238,"they stood before her window, and every year b...",NaN
9239,"roses, white and red.",NaN


Use results to create `DOCS` table.

In [15]:
DOCS = LINES.loc[story_lines].copy()
DOCS['doc_id'] = DOCS['doc_id'].astype(int)
DOCS = DOCS.reset_index().set_index('doc_id')
DOCS = DOCS.rename(columns={'line_str':'doc_title'}).drop('line_num', axis=1)
DOCS

,doc_title
doc_id,
1,THE GOLDEN BIRD
2,HANS IN LUCK
3,JORINDA AND JORINDEL
4,THE TRAVELLING MUSICIANS
5,OLD SULTAN
...,...
59,THE STORY OF THE YOUTH WHO WENT FORTH TO LEARN...
60,KING GRISLY-BEARD
61,IRON HANS


In [16]:
LINES.loc[:, 'doc_id'] = LINES['doc_id'].ffill()
LINES

,line_str,doc_id
line_num,,
121,,NaN
122,,NaN
123,,NaN
124,,NaN
125,THE GOLDEN BIRD,1.0
...,...,...
9237,her children for many years. She took the two ...,63.0
9238,"they stood before her window, and every year b...",63.0
9239,"roses, white and red.",63.0


In [17]:
LINES = LINES.dropna(subset=['doc_id']) # Remove everything before Chapter 1
LINES = LINES.loc[~story_lines] # Remove chapter heading lines; their work is done
LINES.doc_id = LINES.doc_id.astype('int') # Convert chap_num from float to int
LINES.head(100)

,line_str,doc_id
line_num,,
126,,1
127,,1
128,"A certain king had a beautiful garden, and in ...",1
129,which bore golden apples. These apples were al...,1
130,the time when they began to grow ripe it was f...,1
...,...,...
221,"leathern saddle upon him, and not the golden o...",1
222,"Then the son sat down on the fox’s tail, and a...",1
223,and stone till their hair whistled in the wind.,1


In [18]:
OHCO

['doc_id', 'para_num', 'sent_num', 'token_num']

In [19]:
DOCS['doc_str'] = LINES.groupby(OHCO[:1])\
    .line_str.apply(lambda x: '\n'.join(x))\
    .to_frame('doc_str')
DOCS

,doc_title,doc_str
doc_id,,
1,THE GOLDEN BIRD,"\n\nA certain king had a beautiful garden, and..."
2,HANS IN LUCK,\n\nSome men are born to good luck: all they d...
3,JORINDA AND JORINDEL,"\n\nThere was once an old castle, that stood i..."
4,THE TRAVELLING MUSICIANS,\n\nAn honest farmer had once an ass that had ...
5,OLD SULTAN,"\n\nA shepherd had a faithful dog, called Sult..."
...,...,...
59,THE STORY OF THE YOUTH WHO WENT FORTH TO LEARN...,"\n\nA certain father had two sons, the elder o..."
60,KING GRISLY-BEARD,\n\nA great king of a land far away in the Eas...
61,IRON HANS,\n\nThere was once upon a time a king who had ...


In [20]:
# DOCS['doc_str'] = DOCS.doc_str.str.strip().str.replace(r"\n+", " ", regex=True) 
# This was ruining the parsing so I commented it out
DOCS.tail(25)

,doc_title,doc_str
doc_id,,
39,THE JUNIPER-TREE,"\n\nLong, long ago, some two thousand years or..."
40,THE TURNIP,\n\nThere were two brothers who were both sold...
41,CLEVER HANS,"\n\nThe mother of Hans said: ‘Whither away, Ha..."
42,THE THREE LANGUAGES,"\n\nAn aged count once lived in Switzerland, w..."
43,THE FOX AND THE CAT,\n\nIt happened that the cat met the fox in a ...
44,THE FOUR CLEVER BROTHERS,"\n\n‘Dear children,’ said a poor man to his fo..."
45,LILY AND THE LION,"\n\nA merchant, who had three daughters, was o..."
46,THE FOX AND THE HORSE,\n\nA farmer had a horse that had been an exce...
47,THE BLUE LIGHT,\n\nThere was once upon a time a soldier who f...


Save.

In [21]:
DOCS.to_csv(f"../working/pg2591-DOCS.csv", index=True)